# Phase 5 — NB2-Joint: Joint Training Stage 2 (SemEval 2014)

**Goal:** Train Stage 2 with joint training — backprop sentiment loss through embedding model so it learns polarity-consistent retrieval.

**Config:** `stage2_2014_joint.yaml` — joint_training=true, gradient_checkpointing, batch_size=16, grad_accum=4

**Input:**
- `lcminhc/semeval-2014-absa-restaurant` — SemEval 2014 XMLs
- `lcminhc/p5-embed-v4` — embedding ckpt (from NB0)

**Output:** Upload `/kaggle/working/outputs_p5_nb2_joint/` as Kaggle dataset `p5-nb2-joint`
- `stage2_joint_best.pt` (joint model)
- `stage2_joint_best_embedding.pt` (best-epoch embedding)
- `embedding_joint.pt` (final embedding)
- Training logs

## 0. Setup

In [ ]:
!pip install -q transformers faiss-cpu lxml scikit-learn pyyaml

In [ ]:
import os, sys, json, shutil

!git clone https://github.com/lucminhduc3108/Retrieval-ABSA.git /kaggle/working/repo
os.chdir('/kaggle/working/repo')
sys.path.insert(0, '/kaggle/working/repo')
print('Working dir:', os.getcwd())

In [ ]:
# --- Wire SemEval 2014 XMLs ---
KAGGLE_INPUT = None
for candidate in ['/kaggle/input/semeval-2014-absa-restaurant',
                  '/kaggle/input/datasets/lcminhc/semeval-2014-absa-restaurant']:
    if os.path.exists(candidate):
        KAGGLE_INPUT = candidate
        break
assert KAGGLE_INPUT, 'Dataset semeval-2014-absa-restaurant not found'
print(f'XML Input: {KAGGLE_INPUT}')

os.makedirs('SemEval-2014', exist_ok=True)
shutil.copy(f'{KAGGLE_INPUT}/Restaurants_Train.xml',
            'SemEval-2014/Restaurants_Train.xml')
shutil.copy(f'{KAGGLE_INPUT}/Restaurants_Test_Gold.xml',
            'SemEval-2014/Restaurants_Test_Gold.xml')
print('SemEval 2014 XML files wired.')

# --- Prepare data ---
!python scripts/01_prepare_data.py

# --- Wire NB0 embedding checkpoint ---
EMB = None
for candidate in ['/kaggle/input/p5-embed-v4',
                  '/kaggle/input/datasets/lcminhc/p5-embed-v4']:
    if os.path.exists(candidate):
        EMB = candidate
        break
assert EMB, 'Dataset p5-embed-v4 not found'
print(f'\nNB0 Input: {EMB} -> {os.listdir(EMB)}')

os.makedirs('checkpoints/embedding_2014', exist_ok=True)
shutil.copy(f'{EMB}/embedding_v4_s2_best.pt', 'checkpoints/embedding_2014/best.pt')
print(f'Embedding ckpt: {os.path.getsize("checkpoints/embedding_2014/best.pt") / 1e6:.1f} MB')

In [ ]:
import torch, gc
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1. Joint Training — Retrieval (Phase 2a)

Config: `stage2_2014_joint.yaml`
- Joint training: backprop through embedding model
- Per-epoch FAISS index rebuild
- Diagonal W (256 params), tau=0.3, lambda_rank=0.01
- encoder_lr=2e-6, embedding_lr=1e-5, 20 epochs
- batch_size=16, grad_accum=4, gradient_checkpointing=true

In [ ]:
gc.collect()
torch.cuda.empty_cache()
!python scripts/04b_train_stage2.py \
    --config configs/stage2_2014_joint.yaml \
    --embedding_ckpt checkpoints/embedding_2014/best.pt \
    --retrieval_config configs/retrieval_v2.yaml

## 2. Training Log

In [ ]:
log_path = 'logs/stage2_2014_joint_training.jsonl'
print('=== Joint Training ===')
if not os.path.exists(log_path):
    print('No log found.')
else:
    print(f'{"Epoch":<6} {"Loss":<8} {"Acc":<8} {"MacF1":<10} {"pos":<7} {"neg":<7} {"neu":<7}')
    print('-' * 55)
    with open(log_path) as f:
        for line in f:
            r = json.loads(line)
            print(f"{r['epoch']:<6} {r['train_loss']:<8.4f} "
                  f"{r['sentiment_acc']:<8.4f} {r['sentiment_macro_f1']:<10.4f}"
                  f"{r.get('f1_positive', 0):<7.3f} "
                  f"{r.get('f1_negative', 0):<7.3f} "
                  f"{r.get('f1_neutral', 0):<7.3f}")

## 3. Save Outputs

In [ ]:
output_dir = '/kaggle/working/outputs_p5_nb2_joint'
os.makedirs(output_dir, exist_ok=True)
os.makedirs(f'{output_dir}/logs', exist_ok=True)

ckpt_dir = 'checkpoints/stage2_2014_joint'

for src_name, dst_name in [
    ('best.pt', 'stage2_joint_best.pt'),
    ('best_embedding.pt', 'stage2_joint_best_embedding.pt'),
    ('embedding_joint.pt', 'embedding_joint.pt'),
]:
    src = f'{ckpt_dir}/{src_name}'
    if os.path.exists(src):
        shutil.copy(src, f'{output_dir}/{dst_name}')
        print(f'{dst_name}: {os.path.getsize(src)/1e6:.1f} MB')
    else:
        print(f'MISSING: {src}')

log_src = 'logs/stage2_2014_joint_training.jsonl'
if os.path.exists(log_src):
    shutil.copy(log_src, f'{output_dir}/logs/')
    print('Joint training log saved')

print(f'\nOutputs saved to {output_dir}')
print('Upload as Kaggle dataset: p5-nb2-joint')

In [ ]:
shutil.make_archive('/kaggle/working/outputs_p5_nb2_joint_backup', 'zip',
                    '/kaggle/working', 'outputs_p5_nb2_joint')
size_mb = os.path.getsize('/kaggle/working/outputs_p5_nb2_joint_backup.zip') / 1e6
print(f'Backup zip: {size_mb:.1f} MB')